# Sentiment Classification with RoBERTa

This notebook explores the cardiffnlp/twitter-roberta-base-sentiment-latest model.
We run predictions on real reviews and evaluate accuracy against true labels.

Key question: does a model trained on tweets perform well on product reviews?

## 1. Loading Cleaned Sample from Notebook 1

In [2]:
import pandas as pd

df = pd.read_csv('/kaggle/input/notebooks/aliibtissam/01-text-preprocessing/amazon_reviews_cleaned_sample.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print()
print('Sentiment distribution:')
print(df['sentiment'].value_counts())

Shape: (15000, 4)
Columns: ['Text', 'cleaned_text', 'sentiment', 'Score']

Sentiment distribution:
sentiment
positive    5000
neutral     5000
negative    5000
Name: count, dtype: int64


## 2. Loading RoBERTa Model

In [3]:
from transformers import pipeline

# Reload with the correct parameter for this transformers version
classifier = pipeline(
    task='text-classification',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    top_k=None
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

## 3. Running First Sentiment Prediction

In [4]:
review = "This product is absolutely amazing. Best purchase I have ever made."
result = classifier(review)

print('Input:', review)
print()
print('Raw output:')
print(result)
print()
print('Number of scores returned:', len(result[0]))

Input: This product is absolutely amazing. Best purchase I have ever made.

Raw output:
[[{'label': 'positive', 'score': 0.9874584674835205}, {'label': 'neutral', 'score': 0.007520075421780348}, {'label': 'negative', 'score': 0.005021458957344294}]]

Number of scores returned: 3


## 4. Writing a Clean Prediction Function

In [6]:
def predict_sentiment(text):
    # Run the text through RoBERTa
    raw_output = classifier(text)
    
    # raw_output is a list containing one list of three dicts
    # We take index [0] to get the inner list
    scores = raw_output[0]
    
    # Convert to a clean dictionary
    # key = label, value = score
    result = {item['label']: item['score'] for item in scores}
    
    # Find the label with the highest score
    predicted_label = max(result, key=result.get)
    predicted_score = result[predicted_label]
    
    return {
        'label': predicted_label,
        'score': predicted_score,
        'all_scores': result
    }


In [7]:
review = "This product is absolutely amazing. Best purchase I have ever made."
prediction = predict_sentiment(review)

print('Review:', review)
print()
print('Predicted label:', prediction['label'])
print('Confidence:     ', round(prediction['score'], 4))
print()
print('All scores:')
for label, score in prediction['all_scores'].items():
    print(f"  {label:<10} {round(score, 4)}")

Review: This product is absolutely amazing. Best purchase I have ever made.

Predicted label: positive
Confidence:      0.9875

All scores:
  positive   0.9875
  neutral    0.0075
  negative   0.005


## 5. Testing With All Sentiment Types

In [8]:
test_reviews = [
    "This product is absolutely amazing. Best purchase I have ever made.",
    "Terrible quality. Broke after 2 days. Complete waste of money.",
    "It is okay I guess. Does what it says, nothing more nothing less.",
    "I never expected this to be so bad. Very disappointed.",
    "Not great, not terrible. Average product for the price."
]

print(f"{'REVIEW':<60} {'PREDICTED':<12} {'CONFIDENCE'}")
print("-" * 90)

for review in test_reviews:
    prediction = predict_sentiment(review)
    short_review = review[:57] + '...' if len(review) > 57 else review
    print(f"{short_review:<60} {prediction['label']:<12} {round(prediction['score'], 4)}")

REVIEW                                                       PREDICTED    CONFIDENCE
------------------------------------------------------------------------------------------
This product is absolutely amazing. Best purchase I have ... positive     0.9875
Terrible quality. Broke after 2 days. Complete waste of m... negative     0.9503
It is okay I guess. Does what it says, nothing more nothi... neutral      0.6719
I never expected this to be so bad. Very disappointed.       negative     0.9372
Not great, not terrible. Average product for the price.      negative     0.7154


## 6. Original Text vs Cleaned Text Comparison

In [9]:
# Take 5 examples from our dataset
samples = df.head(5)

print(f"{'TEXT TYPE':<12} {'REVIEW':<45} {'TRUE':<10} {'PREDICTED':<10} {'CONFIDENCE'}")
print("-" * 95)

for _, row in samples.iterrows():
    # Predict on original text
    orig_pred = predict_sentiment(row['Text'][:200])
    # Predict on cleaned text
    clean_pred = predict_sentiment(row['cleaned_text'])
    
    short_text = row['Text'][:42] + '...'
    
    print(f"{'original':<12} {short_text:<45} {row['sentiment']:<10} {orig_pred['label']:<10} {round(orig_pred['score'], 4)}")
    print(f"{'cleaned':<12} {'':<45} {row['sentiment']:<10} {clean_pred['label']:<10} {round(clean_pred['score'], 4)}")
    print()

TEXT TYPE    REVIEW                                        TRUE       PREDICTED  CONFIDENCE
-----------------------------------------------------------------------------------------------
original     My 6 month old puppies and adult dogs love... positive   positive   0.9756
cleaned                                                    positive   positive   0.8717

original     I originally purchased the Mauna Loa choco... neutral    neutral    0.8499
cleaned                                                    neutral    neutral    0.4977

original     My Maltese loves these treats and has neve... positive   positive   0.9674
cleaned                                                    positive   positive   0.8981

original     i have no doubt the contents are valuable,... negative   negative   0.4378
cleaned                                                    negative   positive   0.7371

original     these mints were good but not worth the pr... neutral    negative   0.8696
cleaned         

##  Conclusion

RoBERTa performs better on original text than spaCy cleaned text.
Cleaning removes context that RoBERTa relies on for accurate classification.
Final pipeline: original text to RoBERTa, cleaned text to LIME only.

## 7. Running Predictions on Full Sample Dataset

In [10]:
print('Running predictions...')
  
results = []

for text in df['Text']:
    prediction = predict_sentiment(text[:512])
    results.append(prediction['label'])

df['predicted_sentiment'] = results

print('Done.')
print()
print('Prediction distribution:')
print(df['predicted_sentiment'].value_counts())

Running predictions...
Done.

Prediction distribution:
predicted_sentiment
positive    7143
negative    5943
neutral     1914
Name: count, dtype: int64


## 8. Model Evaluation

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

print('Classification Report:')
print(classification_report(df['sentiment'], df['predicted_sentiment']))

Classification Report:
              precision    recall  f1-score   support

    negative       0.63      0.74      0.68      5000
     neutral       0.52      0.20      0.29      5000
    positive       0.62      0.88      0.73      5000

    accuracy                           0.61     15000
   macro avg       0.59      0.61      0.56     15000
weighted avg       0.59      0.61      0.56     15000



In [13]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    df['sentiment'],
    df['predicted_sentiment'],
    labels=['negative', 'neutral', 'positive']
)

print('Confusion Matrix:')
print(f"{'':>12} {'pred_neg':>10} {'pred_neu':>10} {'pred_pos':>10}")
print(f"{'true_neg':>12} {cm[0][0]:>10} {cm[0][1]:>10} {cm[0][2]:>10}")
print(f"{'true_neu':>12} {cm[1][0]:>10} {cm[1][1]:>10} {cm[1][2]:>10}")
print(f"{'true_pos':>12} {cm[2][0]:>10} {cm[2][1]:>10} {cm[2][2]:>10}")

Confusion Matrix:
               pred_neg   pred_neu   pred_pos
    true_neg       3723        587        690
    true_neu       1966        993       2041
    true_pos        254        334       4412


## 9. Testing a Product Review Specific Model

In [14]:
classifier_v2 = pipeline(
    task='text-classification',
    model='lxyuan/distilbert-base-multilingual-cased-sentiments-student',
    top_k=None
)

print('Model v2 loaded successfully.')

config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Model v2 loaded successfully.


## 10. Running Predictions With Product Review Model

In [17]:
def predict_sentiment_v2(text):
    raw_output = classifier_v2(text)
    
    scores = raw_output[0]
    
    result = {item['label']: item['score'] for item in scores}
    
    predicted_label = max(result, key=result.get)
    predicted_score = result[predicted_label]
    
    return {
        'label': predicted_label,
        'score': predicted_score,
        'all_scores': result
    }

In [18]:
test_reviews = [
    "This product is absolutely amazing. Best purchase I have ever made.",
    "Terrible quality. Broke after 2 days. Complete waste of money.",
    "It is okay I guess. Does what it says, nothing more nothing less.",
    "I never expected this to be so bad. Very disappointed.",
    "Not great, not terrible. Average product for the price."
]

print(f"{'REVIEW':<55} {'V1 CARDIFF':<12} {'V2 DISTILBERT'}")
print("-" * 90)

for review in test_reviews:
    v1 = predict_sentiment(review[:512])
    v2 = predict_sentiment_v2(review[:512])
    short = review[:52] + '...'
    print(f"{short:<55} {v1['label']:<12} {v2['label']}")

REVIEW                                                  V1 CARDIFF   V2 DISTILBERT
------------------------------------------------------------------------------------------
This product is absolutely amazing. Best purchase I ... positive     positive
Terrible quality. Broke after 2 days. Complete waste... negative     positive
It is okay I guess. Does what it says, nothing more ... neutral      positive
I never expected this to be so bad. Very disappointe... negative     negative
Not great, not terrible. Average product for the pri... negative     positive


In [19]:
test = classifier_v2("This product is absolutely terrible waste of money.")
print(test)

[[{'label': 'negative', 'score': 0.9706196784973145}, {'label': 'neutral', 'score': 0.01998194307088852}, {'label': 'positive', 'score': 0.009398368187248707}]]


In [20]:
reviews = [
    "This product is absolutely amazing. Best purchase I have ever made.",
    "Terrible quality. Broke after 2 days. Complete waste of money.",
    "It is okay I guess. Does what it says, nothing more nothing less.",
    "I never expected this to be so bad. Very disappointed.",
    "Not great, not terrible. Average product for the price."
]

for review in reviews:
    result = classifier_v2(review)
    print('Review:', review[:50])
    print('Raw output:', result)
    print()

Review: This product is absolutely amazing. Best purchase 
Raw output: [[{'label': 'positive', 'score': 0.9783318638801575}, {'label': 'neutral', 'score': 0.017039505764842033}, {'label': 'negative', 'score': 0.004628594499081373}]]

Review: Terrible quality. Broke after 2 days. Complete was
Raw output: [[{'label': 'positive', 'score': 0.8017158508300781}, {'label': 'negative', 'score': 0.11649321019649506}, {'label': 'neutral', 'score': 0.08179087936878204}]]

Review: It is okay I guess. Does what it says, nothing mor
Raw output: [[{'label': 'positive', 'score': 0.4768970012664795}, {'label': 'neutral', 'score': 0.36853212118148804}, {'label': 'negative', 'score': 0.1545708328485489}]]

Review: I never expected this to be so bad. Very disappoin
Raw output: [[{'label': 'negative', 'score': 0.6788164377212524}, {'label': 'neutral', 'score': 0.2506967782974243}, {'label': 'positive', 'score': 0.07048675417900085}]]

Review: Not great, not terrible. Average product for the p
Raw output: [

In [21]:
from transformers import pipeline

classifier_v3 = pipeline(
    task='text-classification',
    model='yangheng/deberta-v3-base-absa-v1.1',
    top_k=None
)

print('Model v3 loaded successfully.')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: yangheng/deberta-v3-base-absa-v1.1
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Model v3 loaded successfully.


## 10. Testing DeBERTa Model (Product Review Specific)

In [22]:
reviews = [
    "This product is absolutely amazing. Best purchase I have ever made.",
    "Terrible quality. Broke after 2 days. Complete waste of money.",
    "It is okay I guess. Does what it says, nothing more nothing less.",
    "I never expected this to be so bad. Very disappointed.",
    "Not great, not terrible. Average product for the price."
]

for review in reviews:
    result = classifier_v3(review)
    print('Review:', review[:50])
    print('Raw output:', result)
    print()

Review: This product is absolutely amazing. Best purchase 
Raw output: [[{'label': 'Positive', 'score': 0.5753747820854187}, {'label': 'Negative', 'score': 0.34023264050483704}, {'label': 'Neutral', 'score': 0.08439254760742188}]]

Review: Terrible quality. Broke after 2 days. Complete was
Raw output: [[{'label': 'Negative', 'score': 0.7538819313049316}, {'label': 'Positive', 'score': 0.1696922332048416}, {'label': 'Neutral', 'score': 0.07642582058906555}]]

Review: It is okay I guess. Does what it says, nothing mor
Raw output: [[{'label': 'Negative', 'score': 0.5938209295272827}, {'label': 'Positive', 'score': 0.26004984974861145}, {'label': 'Neutral', 'score': 0.14612917602062225}]]

Review: I never expected this to be so bad. Very disappoin
Raw output: [[{'label': 'Positive', 'score': 0.7706884741783142}, {'label': 'Negative', 'score': 0.17709359526634216}, {'label': 'Neutral', 'score': 0.05221788212656975}]]

Review: Not great, not terrible. Average product for the p
Raw output: [[

## 11. Final Model Decision

Tested three models:
- Cardiff RoBERTa (twitter-roberta-base-sentiment-latest) — 61% accuracy
- DistilBERT multilingual (distilbert-base-multilingual-cased-sentiments-student) — poor on English
- DeBERTa ABSA (deberta-v3-base-absa-v1.1) — poor on all classes

Cardiff RoBERTa selected as final model. Best zero-shot performance on Amazon reviews.
61% accuracy is acceptable for a model with no fine-tuning on product review data.

In [23]:
df.to_csv('amazon_reviews_predictions.csv', index=False)
print('Saved successfully.')
print('Columns:', df.columns.tolist())

Saved successfully.
Columns: ['Text', 'cleaned_text', 'sentiment', 'Score', 'predicted_sentiment']
